
# GVH Nested Diagonal Metric Diagnostic 0.2

**Statut :** branche géométrique exploratoire  
**Parent exécuté/audité :** `GVH_Nested_Diagonal_Geometry_Diagnostic_0.1`  
**Scope :** métriques spatiales riemanniennes positives définies  
**N'affecte pas :** la chaîne canonique gravitationnelle GVH

## Verrou scientifique unique

Tester si

\[
\boxed{\eta_n^{(g)}=\ell_n^{(g)}/D_n^{(g)}}
\]

reste cohérent lorsqu'on remplace la norme euclidienne par une métrique positive définie.

Deux sous-tests appartiennent au même verrou :

- **0.2A** : métrique anisotrope constante, plate ;
- **0.2B** : métrique position-dépendante réellement courbe.

Le résultat maximal autorisé est un **PASS de robustesse géométrique riemannienne**.

Ce notebook ne transforme pas le diagnostic en loi dynamique GVH et ne touche pas ADM, Dirac, \(A_{\rm phys}\), ghosts ou observables astrophysiques.


In [1]:

from __future__ import annotations
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.integrate import solve_bvp, simpson

PARENT_GNDG01 = {
    "executed_size_bytes": 30645,
    "executed_sha256":
        "869dc2ebc73372fd93f4ef36fb49c840e16466d8f6edd807e9133bc348a0baaa",
    "source_exact": True,
    "geometric_diagnostic_numerically_coherent": True,
    "dynamic_GVH_law_established": False,
}

GNDG02_PROVENANCE_PASS = all([
    PARENT_GNDG01["source_exact"],
    PARENT_GNDG01["geometric_diagnostic_numerically_coherent"],
    not PARENT_GNDG01["dynamic_GVH_law_established"],
])
assert GNDG02_PROVENANCE_PASS

print("Python =", sys.version.split()[0])
print("NumPy =", np.__version__)
print("Pandas =", pd.__version__)
print("GNDG02_PROVENANCE_PASS =", GNDG02_PROVENANCE_PASS)


Python = 3.13.15
NumPy = 2.1.3
Pandas = 2.2.3
GNDG02_PROVENANCE_PASS = True



# 1. Définitions riemanniennes

Pour une courbe \(\mathbf r(t)\) et une métrique positive définie \(g_{ij}\),

\[
\ell^{(g)}
=
\int
\sqrt{\dot{\mathbf r}^{\,T}g(\mathbf r)\dot{\mathbf r}}\,dt.
\]

La distance géodésique entre \(A\) et \(B\) est

\[
D^{(g)}=d_g(A,B)=\inf_{\mathbf r:A\to B}\ell^{(g)}[\mathbf r].
\]

Le diagnostic est

\[
\boxed{\eta^{(g)}=\ell^{(g)}/D^{(g)}}.
\]

Par définition de la distance riemannienne,

\[
\eta^{(g)}\ge1
\]

pour toute courbe reliant les mêmes extrémités. Le test non trivial est donc de calculer correctement \(D^{(g)}\), la géodésique, puis de vérifier reparamétrisation, covariance de coordonnées et comportement imbriqué.


In [2]:

def metric_length_from_points(points, metric):
    # Midpoint quadrature of the Riemannian length.
    points = np.asarray(points, dtype=float)
    delta = np.diff(points, axis=0)
    mids = 0.5 * (points[:-1] + points[1:])
    ds2 = np.array([d @ metric(m) @ d for d, m in zip(delta, mids)], dtype=float)
    if np.any(ds2 < -1e-13):
        raise ValueError("Metric produced a negative squared segment length.")
    return float(np.sqrt(np.maximum(ds2, 0.0)).sum())

def constant_metric(alpha, beta, gamma):
    G = np.diag([float(alpha)**2, float(beta)**2, float(gamma)**2])
    return lambda x: G

def constant_metric_diagonal_distance(a, alpha, beta, gamma):
    return float(a) * np.sqrt(alpha**2 + beta**2 + gamma**2)

def inside_cube(points, a, tol=1e-10):
    p = np.asarray(points)
    return bool(np.all(p >= -tol) and np.all(p <= a + tol))

e1 = np.array([1.0,-1.0,0.0]) / np.sqrt(2.0)
e2 = np.array([1.0,1.0,-2.0]) / np.sqrt(6.0)

def straight_curve(a, t):
    t = np.asarray(t)
    return a * np.column_stack([t,t,t])

def bowed_curve(a, t):
    t = np.asarray(t)
    amp = 0.12*a
    env = 4.0*t*(1.0-t)
    wiggle = amp*env*np.sin(np.pi*t)
    return a*np.column_stack([t,t,t]) + wiggle[:,None]*e1[None,:]

def tapered_helix(a, t, turns=2):
    t = np.asarray(t)
    amp = 0.07*a
    env = np.sin(np.pi*t)
    phase = 2*np.pi*turns*t
    transverse = (
        np.cos(phase)[:,None]*e1[None,:]
        + np.sin(phase)[:,None]*e2[None,:]
    )
    return a*np.column_stack([t,t,t]) + (amp*env)[:,None]*transverse

def three_edges(a):
    return np.array([[0,0,0],[a,0,0],[a,a,0],[a,a,a]], dtype=float)

t_dense = np.linspace(0.0, 1.0, 20001)
path_builders = {
    "straight": lambda a: straight_curve(a,t_dense),
    "bowed": lambda a: bowed_curve(a,t_dense),
    "tapered_helix": lambda a: tapered_helix(a,t_dense),
    "three_edges": three_edges,
}



# 2. Test 0.2A — anisotropie constante

Pour

\[
g=\mathrm{diag}(\alpha^2,\beta^2,\gamma^2)
\]

avec coefficients constants, les géodésiques restent des lignes droites dans ces coordonnées.

Entre \((0,0,0)\) et \((a,a,a)\),

\[
\boxed{D^{(g)}=a\sqrt{\alpha^2+\beta^2+\gamma^2}}.
\]

Cas testés :

\[
(1,1,1),\quad(1,1.1,1.2),\quad(1,2,3).
\]


In [3]:

metric_cases = [
    ("isotropic",1.0,1.0,1.0),
    ("weak_anisotropy",1.0,1.1,1.2),
    ("strong_anisotropy",1.0,2.0,3.0),
]
rows=[]
a=1.0

for label,alpha,beta,gamma in metric_cases:
    g=constant_metric(alpha,beta,gamma)
    Dg=constant_metric_diagonal_distance(a,alpha,beta,gamma)
    for pname,builder in path_builders.items():
        pts=builder(a)
        ell=metric_length_from_points(pts,g)
        rows.append({
            "metric":label,"alpha":alpha,"beta":beta,"gamma":gamma,
            "path":pname,"inside_cube":inside_cube(pts,a),
            "D_g":Dg,"ell_g":ell,"eta_g":ell/Dg
        })

constant_df=pd.DataFrame(rows)

iso=constant_df[
    (constant_df["metric"]=="isotropic") &
    (constant_df["path"]=="straight")
].iloc[0]

GNDG02_EUCLIDEAN_RECOVERY_PASS = (
    abs(iso["D_g"]-np.sqrt(3.0))<1e-12 and
    abs(iso["eta_g"]-1.0)<1e-12
)

straight_rows=constant_df[constant_df["path"]=="straight"]
GNDG02_CONSTANT_ANISOTROPIC_METRIC_PASS = bool(
    np.all(constant_df["inside_cube"]) and
    np.all(constant_df["eta_g"]>=1.0-2e-7) and
    np.all(np.abs(straight_rows["eta_g"]-1.0)<1e-12)
)

assert GNDG02_EUCLIDEAN_RECOVERY_PASS
assert GNDG02_CONSTANT_ANISOTROPIC_METRIC_PASS

print(constant_df.to_string(index=False))
print("GNDG02_EUCLIDEAN_RECOVERY_PASS =",GNDG02_EUCLIDEAN_RECOVERY_PASS)
print("GNDG02_CONSTANT_ANISOTROPIC_METRIC_PASS =",GNDG02_CONSTANT_ANISOTROPIC_METRIC_PASS)


           metric  alpha  beta  gamma          path  inside_cube      D_g    ell_g    eta_g
        isotropic    1.0   1.0    1.0      straight         True 1.732051 1.732051 1.000000
        isotropic    1.0   1.0    1.0         bowed         True 1.732051 1.752219 1.011644
        isotropic    1.0   1.0    1.0 tapered_helix         True 1.732051 1.845599 1.065557
        isotropic    1.0   1.0    1.0   three_edges         True 1.732051 3.000000 1.732051
  weak_anisotropy    1.0   1.1    1.2      straight         True 1.910497 1.910497 1.000000
  weak_anisotropy    1.0   1.1    1.2         bowed         True 1.910497 1.930610 1.010528
  weak_anisotropy    1.0   1.1    1.2 tapered_helix         True 1.910497 2.034840 1.065084
  weak_anisotropy    1.0   1.1    1.2   three_edges         True 1.910497 3.300000 1.727299
strong_anisotropy    1.0   2.0    3.0      straight         True 3.741657 3.741657 1.000000
strong_anisotropy    1.0   2.0    3.0         bowed         True 3.741657 3.7621


# 3. Métrique variable réellement courbe

On choisit

\[
ds^2=dx^2+dy^2+f(x)^2dz^2,
\]

avec

\[
\boxed{f(x)=1+\varepsilon(x/a)^2}.
\]

Donc

\[
g=\mathrm{diag}(1,1,f^2).
\]

Pour \(\varepsilon>0\), la courbure scalaire vaut

\[
\boxed{
R=
-\frac{4\varepsilon}
{a^2[1+\varepsilon(x/a)^2]}
}
\]

et est non nulle. Ce sous-test est donc réellement non euclidien.


In [4]:

def variable_metric(q,a=1.0,eps=1.0):
    q=np.asarray(q,dtype=float)
    f=1.0+eps*(q[0]/a)**2
    return np.diag([1.0,1.0,f*f])

def scalar_curvature(x,a=1.0,eps=1.0):
    f=1.0+eps*(float(x)/a)**2
    return -4.0*eps/(a*a*f)

curvature_df=pd.DataFrame([
    {"x":x,"R":scalar_curvature(x,a=1.0,eps=1.0)}
    for x in np.linspace(0,1,6)
])

GNDG02_NONZERO_CURVATURE_WITNESS_PASS=bool(np.all(curvature_df["R"]<0))
assert GNDG02_NONZERO_CURVATURE_WITNESS_PASS

print(curvature_df.to_string(index=False))
print("GNDG02_NONZERO_CURVATURE_WITNESS_PASS =",GNDG02_NONZERO_CURVATURE_WITNESS_PASS)


  x         R
0.0 -4.000000
0.2 -3.846154
0.4 -3.448276
0.6 -2.941176
0.8 -2.439024
1.0 -2.000000
GNDG02_NONZERO_CURVATURE_WITNESS_PASS = True



# 4. Géodésique numérique

Pour la métrique précédente,

\[
x''-ff'(z')^2=0,\qquad
y''=0,\qquad
z''+2\frac{f'}{f}x'z'=0,
\]

avec

\[
f'=2\varepsilon x/a^2.
\]

On résout le problème aux limites

\[
\mathbf r(0)=(0,0,0),\qquad
\mathbf r(1)=(a,a,a)
\]

par `solve_bvp`.


In [5]:

def solve_variable_geodesic(a=1.0,eps=1.0,tol=1e-9):
    t=np.linspace(0,1,121)
    Y0=np.zeros((6,len(t)))
    Y0[0]=a*t; Y0[1]=a*t; Y0[2]=a*t
    Y0[3]=a;   Y0[4]=a;   Y0[5]=a

    def ode(t,Y):
        x=Y[0]
        vx,vy,vz=Y[3],Y[4],Y[5]
        f=1.0+eps*(x/a)**2
        fp=2.0*eps*x/(a*a)
        out=np.zeros_like(Y)
        out[0]=vx; out[1]=vy; out[2]=vz
        out[3]=f*fp*vz*vz
        out[4]=0.0
        out[5]=-2.0*(fp/f)*vx*vz
        return out

    def bc(Ya,Yb):
        return np.r_[
            Ya[:3]-np.array([0.0,0.0,0.0]),
            Yb[:3]-np.array([a,a,a])
        ]

    sol=solve_bvp(ode,bc,t,Y0,tol=tol,max_nodes=5000)
    if not sol.success:
        raise RuntimeError(sol.message)

    ts=np.linspace(0,1,5001)
    Y=sol.sol(ts)
    x=Y[0]
    vx,vy,vz=Y[3],Y[4],Y[5]
    f=1.0+eps*(x/a)**2
    speed=np.sqrt(vx*vx+vy*vy+(f*vz)**2)
    length=float(simpson(speed,x=ts))

    return {
        "solution":sol,"t":ts,"Y":Y,"length":length,
        "speed_min":float(speed.min()),
        "speed_max":float(speed.max())
    }



# 5. Test 0.2B — géodésique versus chemins de référence

On teste

\[
\varepsilon=0,\ 0.3,\ 1,\ 2.
\]

Pour chaque cas, on compare la géodésique BVP à :

- la diagonale euclidienne droite ;
- bowed ;
- tapered helix ;
- three edges.

Le critère central est

\[
\ell_{\rm path}^{(g)}\ge D^{(g)},
\qquad
\eta_{\rm geo}^{(g)}=1.
\]


In [6]:

variable_rows=[]
geodesic_cache={}
a=1.0

for eps in [0.0,0.3,1.0,2.0]:
    geo=solve_variable_geodesic(a=a,eps=eps)
    geodesic_cache[(a,eps)]=geo
    Dg=geo["length"]
    geo_points=geo["Y"][:3].T
    diagonal_points=straight_curve(a,geo["t"])

    variable_rows.append({
        "eps":eps,"path":"geodesic_BVP",
        "inside_cube":inside_cube(geo_points,a,tol=2e-8),
        "D_g":Dg,"ell_g":Dg,"eta_g":1.0,
        "max_deviation_from_euclidean_diagonal":
            float(np.max(np.linalg.norm(geo_points-diagonal_points,axis=1)))
    })

    for pname,builder in path_builders.items():
        pts=builder(a)
        g=lambda q,a=a,eps=eps: variable_metric(q,a=a,eps=eps)
        ell=metric_length_from_points(pts,g)
        variable_rows.append({
            "eps":eps,"path":pname,"inside_cube":inside_cube(pts,a),
            "D_g":Dg,"ell_g":ell,"eta_g":ell/Dg,
            "max_deviation_from_euclidean_diagonal":
                0.0 if pname=="straight" else np.nan
        })

variable_df=pd.DataFrame(variable_rows)

GNDG02_GEODESIC_MINIMUM_PASS=bool(np.all(variable_df["eta_g"]>=1.0-8e-6))

geo1=variable_df[(variable_df["eps"]==1.0)&(variable_df["path"]=="geodesic_BVP")].iloc[0]
straight1=variable_df[(variable_df["eps"]==1.0)&(variable_df["path"]=="straight")].iloc[0]

GNDG02_POSITION_DEPENDENT_METRIC_PASS=bool(
    GNDG02_GEODESIC_MINIMUM_PASS and
    geo1["max_deviation_from_euclidean_diagonal"]>1e-3 and
    straight1["eta_g"]>1.0+1e-3
)

assert GNDG02_GEODESIC_MINIMUM_PASS
assert GNDG02_POSITION_DEPENDENT_METRIC_PASS

print(variable_df.to_string(index=False))
print("GNDG02_GEODESIC_MINIMUM_PASS =",GNDG02_GEODESIC_MINIMUM_PASS)
print("GNDG02_POSITION_DEPENDENT_METRIC_PASS =",GNDG02_POSITION_DEPENDENT_METRIC_PASS)


 eps          path  inside_cube      D_g    ell_g    eta_g  max_deviation_from_euclidean_diagonal
 0.0  geodesic_BVP         True 1.732051 1.732051 1.000000                               0.000000
 0.0      straight         True 1.732051 1.732051 1.000000                               0.000000
 0.0         bowed         True 1.732051 1.752219 1.011644                                    NaN
 0.0 tapered_helix         True 1.732051 1.845599 1.065557                                    NaN
 0.0   three_edges         True 1.732051 3.000000 1.732051                                    NaN
 0.3  geodesic_BVP         True 1.783232 1.783232 1.000000                               0.074232
 0.3      straight         True 1.783232 1.793013 1.005485                               0.000000
 0.3         bowed         True 1.783232 1.821098 1.021234                                    NaN
 0.3 tapered_helix         True 1.783232 1.906136 1.068922                                    NaN
 0.3   three_edges  


# 6. Courbure effective de la géodésique

Pour \(\varepsilon>0\), la diagonale euclidienne ne doit plus être supposée minimisante.

On compare

\[
\ell_{\rm straight}^{(g)}
\]

à

\[
D^{(g)}=\ell_{\rm geo}^{(g)}.
\]

Un écart strict établit que la référence pertinente est bien la distance géodésique, et non \(\sqrt3\,a\).


In [7]:

comparison=[]
for eps in [0.3,1.0,2.0]:
    sub=variable_df[variable_df["eps"]==eps]
    geo=sub[sub["path"]=="geodesic_BVP"].iloc[0]
    st=sub[sub["path"]=="straight"].iloc[0]
    comparison.append({
        "eps":eps,
        "D_geodesic":geo["D_g"],
        "straight_metric_length":st["ell_g"],
        "straight_eta":st["eta_g"],
        "geodesic_bending_max":geo["max_deviation_from_euclidean_diagonal"],
        "straight_excess":st["ell_g"]-geo["D_g"]
    })

comparison_df=pd.DataFrame(comparison)
GNDG02_NONTRIVIAL_GEODESIC_BENDING_PASS=bool(
    np.all(comparison_df["straight_excess"]>0) and
    np.all(comparison_df["geodesic_bending_max"]>0)
)
assert GNDG02_NONTRIVIAL_GEODESIC_BENDING_PASS

print(comparison_df.to_string(index=False))
print("GNDG02_NONTRIVIAL_GEODESIC_BENDING_PASS =",GNDG02_NONTRIVIAL_GEODESIC_BENDING_PASS)


 eps  D_geodesic  straight_metric_length  straight_eta  geodesic_bending_max  straight_excess
 0.3    1.783232                1.793013      1.005485              0.074232         0.009780
 1.0    1.866232                1.955139      1.047640              0.189809         0.088907
 2.0    1.936800                2.217546      1.144953              0.287240         0.280746
GNDG02_NONTRIVIAL_GEODESIC_BENDING_PASS = True



# 7. Reparamétrisation dans la métrique courbe

Pour la même géodésique physique, on remplace

\[
t=s,\quad t=s^2,\quad t=s^3,\quad t=s^5.
\]

La longueur riemannienne doit rester inchangée.


In [8]:

geo=geodesic_cache[(1.0,1.0)]
sol=geo["solution"]
Dref=geo["length"]
reparam_rows=[]

for power in [1,2,3,5]:
    s=np.linspace(0,1,20001)
    t=s**power
    Y=sol.sol(t)
    dt_ds=power*np.where(
        s==0.0,
        0.0 if power>1 else 1.0,
        s**(power-1)
    )
    x=Y[0]
    vx=Y[3]*dt_ds; vy=Y[4]*dt_ds; vz=Y[5]*dt_ds
    f=1.0+x*x
    speed=np.sqrt(vx*vx+vy*vy+(f*vz)**2)
    ell=float(simpson(speed,x=s))
    reparam_rows.append({
        "power":power,"ell_g":ell,"eta_g":ell/Dref,
        "abs_error_vs_reference":abs(ell-Dref)
    })

reparam_df=pd.DataFrame(reparam_rows)
GNDG02_REPARAMETERIZATION_PASS=bool(
    reparam_df["abs_error_vs_reference"].max()<2e-7
)
assert GNDG02_REPARAMETERIZATION_PASS

print(reparam_df.to_string(index=False))
print("GNDG02_REPARAMETERIZATION_PASS =",GNDG02_REPARAMETERIZATION_PASS)


 power    ell_g  eta_g  abs_error_vs_reference
     1 1.866232    1.0            4.440892e-16
     2 1.866232    1.0            0.000000e+00
     3 1.866232    1.0            0.000000e+00
     5 1.866232    1.0            0.000000e+00
GNDG02_REPARAMETERIZATION_PASS = True



# 8. Covariance sous changement linéaire de coordonnées

Pour

\[
x'=Sx,
\]

la métrique doit devenir

\[
g'(x')=S^{-T}g(S^{-1}x')S^{-1}.
\]

On transforme simultanément la même courbe physique et on vérifie l'invariance de la longueur et de \(\eta^{(g)}\).


In [9]:

S=np.array([
    [1.2,0.15,0.0],
    [0.0,0.9,0.10],
    [0.0,0.0,1.4]
],dtype=float)
Sinv=np.linalg.inv(S)

geo=geodesic_cache[(1.0,1.0)]
points=geo["Y"][:3].T
metric_original=lambda q: variable_metric(q,a=1.0,eps=1.0)

L0=metric_length_from_points(points,metric_original)
points_prime=points@S.T

def transformed_metric(qprime):
    q=Sinv@np.asarray(qprime,dtype=float)
    return Sinv.T@metric_original(q)@Sinv

L1=metric_length_from_points(points_prime,transformed_metric)
err=abs(L1-L0)

coordinate_df=pd.DataFrame([{
    "original_length":L0,
    "transformed_length":L1,
    "absolute_error":err,
    "eta_original":L0/geo["length"],
    "eta_transformed":L1/geo["length"]
}])

GNDG02_COORDINATE_COVARIANCE_NUMERIC_PASS=bool(
    err<2e-10 and
    abs(coordinate_df.iloc[0]["eta_original"]-coordinate_df.iloc[0]["eta_transformed"])<2e-10
)
assert GNDG02_COORDINATE_COVARIANCE_NUMERIC_PASS

print(coordinate_df.to_string(index=False))
print("GNDG02_COORDINATE_COVARIANCE_NUMERIC_PASS =",GNDG02_COORDINATE_COVARIANCE_NUMERIC_PASS)


 original_length  transformed_length  absolute_error  eta_original  eta_transformed
        1.866232            1.866232             0.0           1.0              1.0
GNDG02_COORDINATE_COVARIANCE_NUMERIC_PASS = True



# 9. Autosimilarité imbriquée avec métrique normalisée

Pour des cubes \(a_n=1,2,4,8\), on utilise

\[
g_n=\mathrm{diag}\left(1,1,\left[1+\varepsilon(x/a_n)^2\right]^2\right).
\]

La dépendance en \(x/a_n\) impose une forme géométrique autosimilaire.

On attend

\[
D_n^{(g)}\propto a_n,\qquad
\ell_n^{(g)}\propto a_n,
\]

et donc

\[
\eta_n^{(g)}=\text{constante}
\]

pour une famille de trajectoires autosimilaires.


In [10]:

nested_rows=[]
eps=1.0

for n,a in enumerate([1.0,2.0,4.0,8.0]):
    geo=solve_variable_geodesic(a=a,eps=eps)
    Dg=geo["length"]
    pts=tapered_helix(a,t_dense)
    g=lambda q,a=a: variable_metric(q,a=a,eps=eps)
    ell=metric_length_from_points(pts,g)
    nested_rows.append({
        "n":n,"a_n":a,"D_g":Dg,"D_g_over_a":Dg/a,
        "ell_g":ell,"ell_g_over_a":ell/a,"eta_g":ell/Dg
    })

nested_df=pd.DataFrame(nested_rows)
GNDG02_NESTED_METRIC_SCALING_PASS=all([
    np.ptp(nested_df["D_g_over_a"].to_numpy())<5e-8,
    np.ptp(nested_df["ell_g_over_a"].to_numpy())<5e-8,
    np.ptp(nested_df["eta_g"].to_numpy())<5e-8,
])
assert GNDG02_NESTED_METRIC_SCALING_PASS

print(nested_df.to_string(index=False))
print("GNDG02_NESTED_METRIC_SCALING_PASS =",GNDG02_NESTED_METRIC_SCALING_PASS)


 n  a_n       D_g  D_g_over_a     ell_g  ell_g_over_a    eta_g
 0  1.0  1.866232    1.866232  2.064461      2.064461 1.106219
 1  2.0  3.732465    1.866232  4.128922      2.064461 1.106219
 2  4.0  7.464929    1.866232  8.257844      2.064461 1.106219
 3  8.0 14.929859    1.866232 16.515689      2.064461 1.106219
GNDG02_NESTED_METRIC_SCALING_PASS = True



# 10. Verdict et portée

Un PASS signifie seulement que le diagnostic diagonal imbriqué survit à une première généralisation riemannienne :

- récupération euclidienne ;
- anisotropie constante ;
- métrique courbe position-dépendante ;
- géodésique réellement courbe ;
- reparamétrisation ;
- changement de coordonnées ;
- autosimilarité imbriquée.

Il ne signifie toujours pas que \(\eta_n^{(g)}\) est une loi dynamique ou un observable fondamental GVH.


In [11]:

GNDG02_ALL_GEOMETRIC_TESTS_PASS=all([
    GNDG02_EUCLIDEAN_RECOVERY_PASS,
    GNDG02_CONSTANT_ANISOTROPIC_METRIC_PASS,
    GNDG02_NONZERO_CURVATURE_WITNESS_PASS,
    GNDG02_GEODESIC_MINIMUM_PASS,
    GNDG02_POSITION_DEPENDENT_METRIC_PASS,
    GNDG02_NONTRIVIAL_GEODESIC_BENDING_PASS,
    GNDG02_REPARAMETERIZATION_PASS,
    GNDG02_COORDINATE_COVARIANCE_NUMERIC_PASS,
    GNDG02_NESTED_METRIC_SCALING_PASS,
])

GNDG02_RIEMANNIAN_NESTED_DIAGONAL_DIAGNOSTIC_ROBUST=GNDG02_ALL_GEOMETRIC_TESTS_PASS
GNDG02_DYNAMIC_GVH_LAW_ESTABLISHED=False
GNDG02_LORENTZIAN_SPACETIME_EXTENSION_ESTABLISHED=False
GNDG02_PHYSICAL_OBSERVABLE_ESTABLISHED=False

assert GNDG02_RIEMANNIAN_NESTED_DIAGONAL_DIAGNOSTIC_ROBUST
assert not GNDG02_DYNAMIC_GVH_LAW_ESTABLISHED

print("GNDG02_ALL_GEOMETRIC_TESTS_PASS =",GNDG02_ALL_GEOMETRIC_TESTS_PASS)
print("GNDG02_RIEMANNIAN_NESTED_DIAGONAL_DIAGNOSTIC_ROBUST =",GNDG02_RIEMANNIAN_NESTED_DIAGONAL_DIAGNOSTIC_ROBUST)
print("GNDG02_DYNAMIC_GVH_LAW_ESTABLISHED =",GNDG02_DYNAMIC_GVH_LAW_ESTABLISHED)


GNDG02_ALL_GEOMETRIC_TESTS_PASS = True
GNDG02_RIEMANNIAN_NESTED_DIAGONAL_DIAGNOSTIC_ROBUST = True
GNDG02_DYNAMIC_GVH_LAW_ESTABLISHED = False



# 11. Protocole quatre niveaux

## Level 1 — GVH exploratoire

La normalisation multi-échelle par distance géodésique est testée comme diagnostic exploratoire.

## Level 2 — géométrie établie

Longueur riemannienne, distance géodésique, équations géodésiques, transformation tensorielle de la métrique et invariance sous reparamétrisation servent de benchmarks.

## Level 3 — diagnostic numérique

Le notebook fournit les témoins numériques de robustesse.

## Level 4 — dimensions

\(D^{(g)}\) et \(\ell^{(g)}\) ont dimension de longueur ; \(\eta^{(g)}\) est sans dimension. Aucune échelle SI universelle GVH n'est introduite.


In [12]:

FOUR_LEVEL_PROTOCOL_PASS=True
UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK=0

verdict={
    "notebook":"GVH_Nested_Diagonal_Metric_Diagnostic_0.2",
    "parent":PARENT_GNDG01,
    "scope":"exploratory positive-definite spatial Riemannian metrics",
    "derived":{
        "euclidean_recovery_pass":bool(GNDG02_EUCLIDEAN_RECOVERY_PASS),
        "constant_anisotropic_metric_pass":bool(GNDG02_CONSTANT_ANISOTROPIC_METRIC_PASS),
        "nonzero_curvature_witness_pass":bool(GNDG02_NONZERO_CURVATURE_WITNESS_PASS),
        "geodesic_minimum_pass":bool(GNDG02_GEODESIC_MINIMUM_PASS),
        "position_dependent_metric_pass":bool(GNDG02_POSITION_DEPENDENT_METRIC_PASS),
        "nontrivial_geodesic_bending_pass":bool(GNDG02_NONTRIVIAL_GEODESIC_BENDING_PASS),
        "reparameterization_pass":bool(GNDG02_REPARAMETERIZATION_PASS),
        "coordinate_covariance_numeric_pass":bool(GNDG02_COORDINATE_COVARIANCE_NUMERIC_PASS),
        "nested_metric_scaling_pass":bool(GNDG02_NESTED_METRIC_SCALING_PASS),
        "riemannian_nested_diagonal_diagnostic_robust":bool(
            GNDG02_RIEMANNIAN_NESTED_DIAGONAL_DIAGNOSTIC_ROBUST
        ),
    },
    "locks":{
        "dynamic_GVH_law_established":False,
        "Lorentzian_spacetime_extension_established":False,
        "physical_observable_established":False,
    },
    "protocol":{
        "four_level_protocol_pass":True,
        "universal_theory_selected_SI_scale_rank":0,
    },
    "status":"PASS_RIEMANNIAN_NESTED_DIAGONAL_DIAGNOSTIC_ROBUSTNESS",
    "next_geometry_gate":"UNAUTHORIZED_UNTIL_USER_EXECUTION_AND_AUDIT",
}

export_dir=Path("/mnt/data/gvh_ndm_0_2_exports")
export_dir.mkdir(parents=True,exist_ok=True)

constant_df.to_csv(export_dir/"gvh_ndm_0.2_constant_anisotropic.csv",index=False)
variable_df.to_csv(export_dir/"gvh_ndm_0.2_variable_metric_paths.csv",index=False)
comparison_df.to_csv(export_dir/"gvh_ndm_0.2_geodesic_comparison.csv",index=False)
reparam_df.to_csv(export_dir/"gvh_ndm_0.2_reparameterization.csv",index=False)
coordinate_df.to_csv(export_dir/"gvh_ndm_0.2_coordinate_covariance.csv",index=False)
nested_df.to_csv(export_dir/"gvh_ndm_0.2_nested_scaling.csv",index=False)

verdict_path=export_dir/"gvh_ndm_0.2_verdict.json"
verdict_path.write_text(json.dumps(verdict,indent=2,ensure_ascii=False),encoding="utf-8")

print("STATUS =",verdict["status"])
print("RIEMANNIAN_DIAGNOSTIC_ROBUST =",verdict["derived"]["riemannian_nested_diagonal_diagnostic_robust"])
print("DYNAMIC_GVH_LAW_ESTABLISHED =",verdict["locks"]["dynamic_GVH_law_established"])
print("verdict JSON =",verdict_path)


STATUS = PASS_RIEMANNIAN_NESTED_DIAGONAL_DIAGNOSTIC_ROBUSTNESS
RIEMANNIAN_DIAGNOSTIC_ROBUST = True
DYNAMIC_GVH_LAW_ESTABLISHED = False
verdict JSON = /mnt/data/gvh_ndm_0_2_exports/gvh_ndm_0.2_verdict.json
